In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

In [2]:
import requests

# 1. Define tu token personal (reemplaza 'TU_TOKEN' con el código de tu cuenta del Banco Central)
mi_token = '$2a$10$mpHkh9mZh.qYC/3jTxDsjuYnkJBSQAxfBS1Ckw.e3Wo3hJrL7KscK'

# 2. Crea una función para no repetir código por cada dato que necesites
def obtener_dato_bcentral(codigo_serie, token):
    # Armar el enlace de la API
    url = f"https://si3.bcentral.cl/SieteRestWS/SieteRestWS.ashx?token={token}&timeseries={codigo_serie}&function=GetSeries"
    
    # Hacer la petición web
    respuesta = requests.get(url)
    datos = respuesta.json()
    
    # Extraer las observaciones del JSON a un DataFrame
    df = pd.DataFrame(datos['Series']['Obs'])
    
    # Formatear las columnas: convertir fecha a datetime y el valor a número
    df['fecha'] = pd.to_datetime(df['indexDateString'], format='%d-%m-%Y')
    df['valor'] = pd.to_numeric(df['value'], errors='coerce')
    
    # Filtrar solo las columnas útiles y renombrar el valor con el nombre de la serie
    df = df[['fecha', 'valor']].rename(columns={'valor': codigo_serie})
    
    return df

# 3. Descargar las dos series que necesitas usando sus códigos API
# (Uso los códigos de tu imagen como ejemplo)
df_ipc_general = obtener_dato_bcentral('F074.IPC.IND.Z.EP23.C.M', mi_token)
df_ipc_alimentos = obtener_dato_bcentral('F074.IPCA.IND.Z.EP23.Z.M', mi_token)

# 4. Unir ambas bases de datos respetando todas las fechas (outer join)
df_maestra = pd.merge(df_ipc_general, df_ipc_alimentos, on='fecha', how='outer')

# 5. Ordenar por fecha para mayor claridad
df_maestra = df_maestra.sort_values('fecha').reset_index(drop=True)

print(df_maestra)

         fecha  F074.IPC.IND.Z.EP23.C.M  F074.IPCA.IND.Z.EP23.Z.M
0   2009-12-01                    57.31                     42.65
1   2010-01-01                    57.61                     42.39
2   2010-02-01                    57.77                     42.31
3   2010-03-01                    57.82                     42.69
4   2010-04-01                    58.09                     42.60
..         ...                      ...                       ...
197 2026-05-01                   112.37                    110.74
198 2026-06-01                   112.35                    111.60
199 2026-07-01                   112.45                    112.37
200 2026-08-01                   113.15                    113.98
201 2026-09-01                   113.59                    114.66

[202 rows x 3 columns]


In [14]:
df_maestra = df_maestra.rename(columns={
    'F074.IPC.IND.Z.EP23.C.M': 'ipc',
    'F074.IPCA.IND.Z.EP23.Z.M': 'ipc_alimentos'
})

print(df_maestra)

         fecha     ipc  ipc_alimentos
0   2009-12-01   57.31          42.65
1   2010-01-01   57.61          42.39
2   2010-02-01   57.77          42.31
3   2010-03-01   57.82          42.69
4   2010-04-01   58.09          42.60
..         ...     ...            ...
197 2026-05-01  112.37         110.74
198 2026-06-01  112.35         111.60
199 2026-07-01  112.45         112.37
200 2026-08-01  113.15         113.98
201 2026-09-01  113.59         114.66

[202 rows x 3 columns]
